In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv
from typing import TypedDict, List, Optional, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from schemas import State, EvidencePack
from langchain_ollama import ChatOllama

load_dotenv()

True

In [10]:
# tool = TavilySearchResults(max_results=2)
# results = tool.invoke({'query': 'trends in mern stack technology'})

# results

In [ ]:

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.3,
)

In [11]:
# for r in results:
#     print(r['content'])

In [12]:
def __tavily_search_(query: str, max_results: int = 5) -> List[dict]:

    tool = TavilySearchResults(max_results=max_results)
    results = tool.invoke({'query': query})

    # create an empty list of dict stored in var named normalized
    normalized = List[dict] = []

    for r in results or []:
        normalized.append(
            {
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": r.get("content") or r.get("snippet") or "",
                "published_at": r.get("published_date") or r.get("published_at"),
                "source": r.get("source"),
            }
        )

    return normalized

In [13]:
RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""

In [14]:
def search_tool(state: State) -> dict:

    # take the fist 10 queries from state
    queries = (state.get("queries" , []) or [])
    max_results = 6

    raw_results: List[dict] = []

    for q in queries:
        raw_results.extend(__tavily_search_(q, max_results=max_results))

    if not raw_results:
        return {"evidence": []}
    
 
    extractor = llm.with_structured_output(EvidencePack)
    pack = extractor.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw results:\n{raw_results}"),
        ]
    )

    # Deduplicate by URL
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e

    return {"evidence": list(dedup.values())}
